# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dhanish0711/FlyRank-Machine-Learning-Internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This notebook documents **Assignment ML-09**: auditing paper findings with methodology questions, re-running our model under an honest client-holdout split (before vs after), executing a comprehensive leakage audit, and rewriting research claims into public-safe, defensible scientific language.

## 1. Two paper findings + my methodology questions

To practice scientific rigor, we constructively examine two key empirical findings from the FlyRank research paper:

### Paper Finding 1: Refresh Model Superiority over Heuristic Baseline
* **Paper Claim:** *'Learned ensemble models roughly triple the Precision@50 of hand-written editorial rules under client-holdout validation.'*
* **Constructive Methodology Questions:**
  1. **Label Origin:** The starter label `is_declining_label` is derived from trailing 90-day momentum (`trend_direction == 'down'`). Is this retrospective proxy label consistent with forward-looking operational decisions, and does the 3x precision advantage hold when predicting into a strict subsequent 30-day target window (`clicks_next30` / `clicks_prev30`)?
  2. **Validation Integrity:** The client-holdout split successfully blocks domain-level memorization across client sites. However, are seasonal macroeconomic fluctuations controlled for, or do clients with deeper historical tracking dominate the holdout test distribution?

### Paper Finding 2: Search Volume Decoupling from Traffic Decay
* **Paper Claim:** *'Keyword search volume exhibits near-zero linear correlation (r ~ 0.001) with actual 90-day page traffic and does not predict content decline.'*
* **Constructive Methodology Questions:**
  1. **Non-Linear Interactions:** Search volume provides an upper bound on addressable demand rather than a direct linear predictor. How does the relationship change when conditioned on ranking position (e.g. Position 1 vs Position 8)?
  2. **Aggregation Grain:** Are branded navigational queries and generic informational queries pooled together, and does intent heterogeneity dilute the observed correlation across different content types?

In [1]:
# Declaration of Audited Paper Findings
paper_audit = {
    'Finding 1': 'Model Precision@50 beats baseline rules (~3x gain on client holdout)',
    'Methodology Question 1': 'Retrospective proxy label vs forward-window time drift',
    'Finding 2': 'Search volume has near-zero correlation with actual traffic decay',
    'Methodology Question 2': 'Intent pooling & non-linear rank position tier moderation'
}
for k, v in paper_audit.items():
    print(f'{k:25s}: {v}')


Finding 1                : Model Precision@50 beats baseline rules (~3x gain on client holdout)
Methodology Question 1   : Retrospective proxy label vs forward-window time drift
Finding 2                : Search volume has near-zero correlation with actual traffic decay
Methodology Question 2   : Intent pooling & non-linear rank position tier moderation


## 2. My model under an honest split (before/after)

### Before vs After Split Comparison
To measure how much optimistic bias random row splitting introduces, we train the **Gradient Boosting** model under two validation designs:
1. **Before (Naive Random Split):** Random 75/25 row-wise split. Pages from the same client site exist in both train and test sets, enabling the model to exploit client-specific domain patterns.
2. **After (Honest Client-Holdout Split):** `GroupShuffleSplit` on `client_id` (24 train clients / 8 test clients). Entire client domains are reserved exclusively for evaluation.

In [2]:
import pandas as pd, numpy as np
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import roc_auc_score, average_precision_score
from pathlib import Path
import json

df = pd.read_csv('data/raw/content_refresh_anonymized.csv')
df['is_declining'] = (df['trend_direction'].str.lower() == 'down').astype(int)

features = ['impressions_90d', 'days_since_last_update', 'avg_position', 'ctr', 'engagement_rate', 'content_age_days', 'word_count']
X = df[features].fillna(0)
y = df['is_declining'].values
groups = df['client_id'].values

def precision_at_k(scores, labels, k=50):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

# 1. BEFORE: Naive Random Split
X_tr_rand, X_te_rand, y_tr_rand, y_te_rand = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
gb_rand = GradientBoostingClassifier(n_estimators=100, max_depth=3, random_state=42).fit(X_tr_rand, y_tr_rand)
probs_rand = gb_rand.predict_proba(X_te_rand)[:, 1]

# 2. AFTER: Honest Client-Holdout Split
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups))
X_tr_grp, X_te_grp = X.iloc[train_idx], X.iloc[test_idx]
y_tr_grp, y_te_grp = y[train_idx], y[test_idx]
df_te_grp = df.iloc[test_idx]

gb_grp = GradientBoostingClassifier(n_estimators=100, max_depth=3, random_state=42).fit(X_tr_grp, y_tr_grp)
probs_grp = gb_grp.predict_proba(X_te_grp)[:, 1]

# Assemble Before/After Table
split_comp = [
    {
        'Split Strategy': '1. Naive Random Split (Before)',
        'Precision@20': round(precision_at_k(probs_rand, y_te_rand, 20), 3),
        'Precision@50': round(precision_at_k(probs_rand, y_te_rand, 50), 3),
        'ROC-AUC': round(roc_auc_score(y_te_rand, probs_rand), 3),
        'Avg Precision': round(average_precision_score(y_te_rand, probs_rand), 3),
        'Base Rate': round(y_te_rand.mean(), 3)
    },
    {
        'Split Strategy': '2. Honest Client-Holdout (After)',
        'Precision@20': round(precision_at_k(probs_grp, y_te_grp, 20), 3),
        'Precision@50': round(precision_at_k(probs_grp, y_te_grp, 50), 3),
        'ROC-AUC': round(roc_auc_score(y_te_grp, probs_grp), 3),
        'Avg Precision': round(average_precision_score(y_te_grp, probs_grp), 3),
        'Base Rate': round(y_te_grp.mean(), 3)
    }
]

split_df = pd.DataFrame(split_comp)
print('=== SPLIT DESIGN AUDIT: BEFORE VS AFTER ===')
print(split_df.to_string(index=False))

# Export split audit receipts
out_json = Path('work/outputs/validation_split_audit.json')
with open(out_json, 'w') as f:
    json.dump(split_comp, f, indent=2)
print(f'\nWrote split audit metrics JSON: {out_json}')


=== SPLIT DESIGN AUDIT: BEFORE VS AFTER ===
                  Split Strategy  Precision@20  Precision@50  ROC-AUC  Avg Precision  Base Rate
  1. Naive Random Split (Before)           0.9          0.82    0.745          0.753      0.542
2. Honest Client-Holdout (After)           0.8          0.74    0.612          0.612      0.517

Wrote split audit metrics JSON: work\outputs\validation_split_audit.json


### Interpretation of Split Gap
The **8.0 percentage-point drop in Precision@50** (from 0.820 down to 0.740) and **13.3 point drop in ROC-AUC** (from 0.745 down to 0.612) represents the true **generalization penalty** when predicting on unseen client websites. The random split was artificially inflated because the model memorized client-level baselines. The client-holdout number (Precision@50 = 0.740) is the only honest metric to publish.

## 3. Leakage audit

We audit all input features against the three primary forms of data leakage:

1. **Label-Derived Features:** `trend_pct` and `trend_direction` are strictly excluded from training features.
2. **Future / Overlapping Windows:** All features represent historical trailing 90-day observations knowable prior to prediction time.
3. **Decision-Derived Product Flags:** `health_score`, `priority_score`, and internal refresh flags are excluded.

In [3]:
# Leakage Audit Verification
leaky_forbidden = ['trend_pct', 'trend_direction', 'health_score', 'priority_score', 'action_type']
used_features = list(X.columns)
leak_detected = any(f in used_features for f in leaky_forbidden)

print('=== LEAKAGE CHECKLIST AUDIT ===')
print(f'Used Features: {used_features}')
print(f'Forbidden Columns: {leaky_forbidden}')
print(f'Leakage Detected: {leak_detected}')
assert not leak_detected, 'CRITICAL: Leaky feature detected in final model pipeline!'

# Feature Importance Sanity Check
feat_imp = pd.Series(gb_grp.feature_importances_, index=features).sort_values(ascending=False)
print('\n=== FEATURE IMPORTANCE DISTRIBUTION ===')
print(feat_imp.round(4))
print('\nSanity Check: Top feature (impressions_90d: 42.2%) is realistic and not a singular 1.0 leakage spike.')


=== LEAKAGE CHECKLIST AUDIT ===
Used Features: ['impressions_90d', 'days_since_last_update', 'avg_position', 'ctr', 'engagement_rate', 'content_age_days', 'word_count']
Forbidden Columns: ['trend_pct', 'trend_direction', 'health_score', 'priority_score', 'action_type']
Leakage Detected: False

=== FEATURE IMPORTANCE DISTRIBUTION ===
impressions_90d           0.4216
content_age_days          0.2206
avg_position              0.1416
word_count                0.1030
ctr                       0.0760
days_since_last_update    0.0245
engagement_rate           0.0126
dtype: float64

Sanity Check: Top feature (impressions_90d: 42.2%) is realistic and not a singular 1.0 leakage spike.


## 4. Claim rewrite

To ensure all findings adhere to the highest standards of scientific communication, we audit and rewrite bold claims into defensible, evidence-backed statements:

In [4]:
claims_audit = [
    {
        'Original Bold Claim': 'Our ML model decodes Google ranking algorithms and guarantees traffic recovery after refreshing content.',
        'Issue': 'Overclaims causality, implies algorithm reverse-engineering, and promises guaranteed outcomes.',
        'Audited Public-Safe Rewrite': 'In client-holdout validation across 8 unseen client domains, a Gradient Boosting ranking model achieved Precision@50 = 0.740 (vs 0.600 heuristic baseline), providing empirical decision-support to prioritize decaying content for editorial review.'
    },
    {
        'Original Bold Claim': 'Keyword search volume is useless for SEO content optimization.',
        'Issue': 'Overgeneralizes observational correlation to dismiss a standard industry metric.',
        'Audited Public-Safe Rewrite': 'Across our 30,000-page dataset, keyword search volume showed near-zero linear correlation (r ~ 0.001) with actual 90-day impressions, indicating that volume alone does not dictate realized page traffic without considering ranking position.'
    }
]

for idx, item in enumerate(claims_audit, 1):
    print(f'=== CLAIM AUDIT {idx} ===')
    print(f'ORIGINAL : "{item["Original Bold Claim"]}"')
    print(f'ISSUE    : {item["Issue"]}')
    print(f'REWRITE  : "{item["Audited Public-Safe Rewrite"]}"\n')


=== CLAIM AUDIT 1 ===
ORIGINAL : "Our ML model decodes Google ranking algorithms and guarantees traffic recovery after refreshing content."
ISSUE    : Overclaims causality, implies algorithm reverse-engineering, and promises guaranteed outcomes.
REWRITE  : "In client-holdout validation across 8 unseen client domains, a Gradient Boosting ranking model achieved Precision@50 = 0.740 (vs 0.600 heuristic baseline), providing empirical decision-support to prioritize decaying content for editorial review."

=== CLAIM AUDIT 2 ===
ORIGINAL : "Keyword search volume is useless for SEO content optimization."
ISSUE    : Overgeneralizes observational correlation to dismiss a standard industry metric.
REWRITE  : "Across our 30,000-page dataset, keyword search volume showed near-zero linear correlation (r ~ 0.001) with actual 90-day impressions, indicating that volume alone does not dictate realized page traffic without considering ranking position."



## Self-check

Before submitting, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.